In [1]:
"""
Project: Loan Approval Prediction Engine
Author: Eyar Yaish

Purpose: 
This script processes financial dataset records to train a machine learning model 
capable of predicting loan default risk and providing calibrated probability scores.

Architecture: 
The engine uses a scikit-learn Pipeline that combines a StandardScaler (for Z-score 
normalization) and a Support Vector Classifier (SVC) utilizing an RBF kernel mathematically 
optimized for handling imbalanced financial data.

Expected Output: 
Executes model training, outputs evaluation metrics (including the ROC AUC score), 
and saves the finalized production model as a 'loan_svc_project_2.pkl' file to be 
deployed via a Flask REST API.
"""

"\nProject: Loan Approval Prediction Engine\nAuthor: Eyar Yaish\n\nPurpose: \nThis script processes financial dataset records to train a machine learning model \ncapable of predicting loan default risk and providing calibrated probability scores.\n\nArchitecture: \nThe engine uses a scikit-learn Pipeline that combines a StandardScaler (for Z-score \nnormalization) and a Support Vector Classifier (SVC) utilizing an RBF kernel mathematically \noptimized for handling imbalanced financial data.\n\nExpected Output: \nExecutes model training, outputs evaluation metrics (including the ROC AUC score), \nand saves the finalized production model as a 'loan_svc_project_2.pkl' file to be \ndeployed via a Flask REST API.\n"

In [2]:
# Standard Environment Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, clear_output
from rich import print

# Support Vector engines
from sklearn.svm import SVC, SVR
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RandomizedSearchCV

# pipline and file making
import joblib
from sklearn.pipeline import Pipeline

# Evaluation Metrics (Choose based on task)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,classification_report
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import roc_auc_score, roc_curve
%config Completer.use_jedi = False

In [3]:
# remaking the table- using only 6 needed col's.
try:
    Loan_Prediction = pd.read_csv('Loan_approval_data_2025.csv')

    Loan_Prediction_clean = Loan_Prediction[[
        'loan_status',
        'annual_income',
        'loan_amount',
        'credit_score',
        'debt_to_income_ratio',
        'years_employed',
        'delinquencies_last_2yrs'
    ]]
    display('defined fitures for Loan_Prediction DF:', Loan_Prediction_clean)

    display(Loan_Prediction_clean.info(show_counts=False))   # all row's are numeric so no need to toch the data
    print(Loan_Prediction_clean.isna().sum())                # no NuN row's so no need for dropna or refilling data

except Exception as e:
    print(f"Error reading file: {e}")

'defined fitures for Loan_Prediction DF:'

,loan_status,annual_income,loan_amount,credit_score,debt_to_income_ratio,years_employed,delinquencies_last_2yrs
0,1,25579,600,692,0.423,17.2,0
1,0,43087,53300,627,0.384,7.3,1
2,1,20840,2100,689,0.377,1.1,0
3,1,29147,2900,692,0.398,0.5,1
4,1,63657,99600,630,0.195,12.5,0
...,...,...,...,...,...,...,...
49995,0,39449,42800,570,0.192,4.3,0
49996,0,20496,3800,672,0.306,4.4,0
49997,0,18743,18000,719,0.551,4.8,0
49998,0,17250,1400,633,0.451,0.4,0


<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 7 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   loan_status              int64  
 1   annual_income            int64  
 2   loan_amount              int64  
 3   credit_score             int64  
 4   debt_to_income_ratio     float64
 5   years_employed           float64
 6   delinquencies_last_2yrs  int64  
dtypes: float64(2), int64(5)
memory usage: 2.7 MB


None

loan_status                0
annual_income              0
loan_amount                0
credit_score               0
debt_to_income_ratio       0
years_employed             0
delinquencies_last_2yrs    0
dtype: int64

In [ ]:
# reformating the paremetters into a "float32" DTYPE to minimize the ram usage (no harm to data)

Loan_Prediction_clean = Loan_Prediction_clean.astype('float32')

# making the Pipeline, it is using a rbf model chosen affter analysis.
pipeline = Pipeline([
    ('scaler', StandardScaler()), 
    ('svc', SVC(random_state=42))
])

# exstracting the fitures and labels
Y_label = Loan_Prediction_clean['loan_status']
X_fiture = Loan_Prediction_clean[[
        'annual_income',
        'loan_amount',
        'credit_score',
        'debt_to_income_ratio',
        'years_employed',
        'delinquencies_last_2yrs'
    ]]

X_train, X_test, y_train, y_test = train_test_split(
    X_fiture, Y_label, test_size=0.2, random_state=42)

# for the rbf model we will need to find 3 sub-parameters - 
    # 1. class_weight- a grading system to prevent a lazy model ffrom guessing randomly
    # C- asigned penalty for mistakes. prevents over and under fitting the model so patterns could imarge.
    # gamma- to adjust the influense of each point on the the plane (6D).
           # so each point wont over (or under) influenses the models structure.

weights = [{0: weight_val, 1: weight_val} for weight_val in np.linspace(0.1, 5, 10)] # defining the value of the weight's/


param_grid = {
        'svc__kernel': ['rbf'],
        'svc__C': np.linspace(5, 8, 10),
        'svc__gamma': np.linspace(0.1, 2, 10),
        'svc__class_weight': weights
    }                                            # creating the base for the gridsearch in compliance for RBF model 


# the main grid engine - searcing throw the grid for 100 exempels to test, the gridsearc will comput for a total of 300 laps.  
search = RandomizedSearchCV(
    estimator = pipeline,
    param_distributions = param_grid,
    n_iter = 60,                  
    scoring = 'roc_auc',      
    cv = 3,                       
    n_jobs =- 1,                  
    verbose = 3,                  
    random_state = 42)





search.fit(X_train, y_train) #the "START button" will start the traning. 

# scores outputs
print("Winning Score:", search.best_score_)
print("Winning Parameters:", search.best_params_)

best_loan_estimator = search.best_estimator_
model_filename = 'loan_svc_project.pkl'
joblib.dump(best_loan_estimator, model_filename)

print(f"Success! Model securely saved as: {model_filename}")

Fitting 3 folds for each of 100 candidates, totalling 300 fits


Winning Score: 0.9014835378365356

Winning Parameters:
{
    'svc__kernel': 'rbf',
    'svc__gamma': np.float64(0.8),
    'svc__class_weight': {0: np.float64(1.0), 1: 1.0},
    'svc__C': np.float64(7.308333333333332)
}

In [12]:
# reformating the paremetters into a "float32" DTYPE to minimize the ram usage (no harm to data)

Loan_Prediction_clean = Loan_Prediction_clean.astype('float32')

# making the Pipeline, it is using a rbf model chosen affter analysis.
class_map = {0: 4.2, 1: 2.2}
pipeline = Pipeline([
    ('scaler', StandardScaler()), 
    ('svc', SVC(class_weight=class_map, C=7.308, gamma=0.2, probability=True, random_state=42))
])


# exstracting the fitures and labels
Y_label = Loan_Prediction_clean['loan_status']
X_fiture = Loan_Prediction_clean[[
        'annual_income',
        'loan_amount',
        'credit_score',
        'debt_to_income_ratio',
        'years_employed',
        'delinquencies_last_2yrs'
    ]]
X_train, X_test, y_train, y_test = train_test_split(
    X_fiture, Y_label, test_size=0.2, random_state=42)   # creating 2 saperrated fitures and labels.
                                                         # one for the training the other for evaluation of trained model.


# for the rbf model we will need to find 3 sub-parameters - 
    # 1. class_weight- a grading system to prevent a lazy model ffrom guessing randomly
    # C- asigned penalty for mistakes. prevents over and under fitting the model so patterns could imarge.
    # gamma- to adjust the influense of each point on the the plane (6D).
           # so each point wont over (or under) influenses the models structure.

scores = cross_val_score(
    pipeline,
    X_train, y_train,
    cv = 3,
    scoring = ('roc_auc'))


print("Scores:", scores)
print(f"\nAverage ROC AUC: {scores.mean():.2f}")

pipeline.fit(X_train, y_train)  # the "START button" will start the traning. 

y_pred = (pipeline.predict_proba(X_test)[:, 1] >= 0.75).astype(int)  # trying to predict the accurate lables

# analysis of prediction quality:
y_proba = pipeline.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_proba)
print(f"Held-out Test ROC AUC: {test_auc:.4f}")

cm = confusion_matrix(y_test, y_pred)

print("Confusion matrix:\n", cm)
print("Classification Report:\n", classification_report(y_test, y_pred))

# creating the pkl file extracted from the pipeilne (trained) model we had created above.
model_filename = 'loan_svc_project_2.pkl'
joblib.dump(pipeline, model_filename)

print(f"Success! Model securely saved as: {model_filename}")

C:\Users\eyar1\anaconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\eyar1\anaconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\eyar1\anaconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Scores: [0.91886    0.91667149 0.91789659]

Average ROC AUC: 0.92

C:\Users\eyar1\anaconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Held-out Test ROC AUC: 0.9146

Confusion matrix:
 [[4158  351]
 [1591 3900]]

Classification Report:
               precision    recall  f1-score   support

         0.0       0.72      0.92      0.81      4509
         1.0       0.92      0.71      0.80      5491

    accuracy                           0.81     10000
   macro avg       0.82      0.82      0.81     10000
weighted avg       0.83      0.81      0.81     10000

Success! Model securely saved as: loan_svc_project_2.pkl

In [13]:
# 1. Loading the Model
model_filename = 'loan_svc_project_2.pkl'
pipeline = joblib.load(model_filename)

# 2. Function: Get the Features List
def get_model_features():
    """Returns the list of features the model was trained on."""
    return [
        'annual_income', 
        'loan_amount', 
        'credit_score', 
        'debt_to_income_ratio', 
        'years_employed', 
        'delinquencies_last_2yrs'
    ]

# 3. Function: Get Sample Data
def get_sample_data(df, num_rows=3):
    """Takes your DataFrame and returns a few rows as sample data."""
    # .to_dict('records') formats the data perfectly for web APIs
    return df.head(num_rows).to_dict(orient='records')

# 4. Function: Get Margin and Calculation Function Info
def get_margin_and_calc_info():
    """Extracts the RBF mathematical parameters from the SVC model."""
    # Extract the SVC engine from the pipeline
    svc_engine = pipeline.named_steps['svc']
    
    # Get the number of support vectors (which define the RBF margin)
    total_support_vectors = int(sum(svc_engine.n_support_))
    intercept = float(svc_engine.intercept_[0])
    
    return {
        "kernel_type": svc_engine.kernel,
        "calculation_function": "sum(alpha_i * y_i * RBF_Kernel(x_i, x)) + b",
        "intercept_b": intercept,
        "total_support_vectors": total_support_vectors,
        "margin_explanation": "Because this is an RBF kernel, the margin is non-linear and defined by the multidimensional support vectors."
    }

# 5. Function: Get General Model Metadata
def get_model_metadata(roc_auc_score):
    """Returns the file name and accuracy of the model."""
    return {
        "file_name": model_filename,
        "model_accuracy_roc_auc": round(roc_auc_score, 4),
        "status": "Ready for Production"
    }

# --- TESTING THE FUNCTIONS ---
print("--- Features ---")
print(get_model_features())

print("\n--- Margin & Calculation Info ---")
print(get_margin_and_calc_info())

# Assuming your mean cross-val score from earlier was 0.90
print("\n--- Model Metadata ---")
print(get_model_metadata(0.90))

print("\n--- Sample Data ---")
# Pass your X_fiture variable into this function to test it
print(get_sample_data(X_fiture, 1))

--- Features ---

[
    'annual_income',
    'loan_amount',
    'credit_score',
    'debt_to_income_ratio',
    'years_employed',
    'delinquencies_last_2yrs'
]

--- Margin & Calculation Info ---

{
    'kernel_type': 'rbf',
    'calculation_function': 'sum(alpha_i * y_i * RBF_Kernel(x_i, x)) + b',
    'intercept_b': -1.342198468071152,
    'total_support_vectors': 14794,
    'margin_explanation': 'Because this is an RBF kernel, the margin is non-linear and defined by the 
multidimensional support vectors.'
}

--- Model Metadata ---

{'file_name': 'loan_svc_project_2.pkl', 'model_accuracy_roc_auc': 0.9, 'status': 'Ready for Production'}

--- Sample Data ---

[
    {
        'annual_income': 25579.0,
        'loan_amount': 600.0,
        'credit_score': 692.0,
        'debt_to_income_ratio': 0.4230000078678131,
        'years_employed': 17.200000762939453,
        'delinquencies_last_2yrs': 0.0
    }
]